# Dataset Real Estate Listings (Data.World)

## Creating Folders

In [ ]:
!echo "Creating data folder"
!mkdir -p data/bronze data/silver data/gold

## Downloading Dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
DATASET_LINK = "https://query.data.world/s/3ox2v5trynvowrhc27fjqkcx3lytjf?dws=00000"
DATASET_NAME = "properati-BR-2016-11-01-properties-rent"

In [ ]:
# df = pd.read_csv(DATASET_LINK)
# df.to_csv(f"data/bronze/{DATASET_NAME}.csv", index=False)

In [ ]:
df = pd.read_csv(f'data/bronze/{DATASET_NAME}.csv')

In [ ]:
df.info()

In [ ]:
df.head()

## Cleaning (Silver)

### Split 'place' column

In [ ]:
def split_place_with_parent_names(src_df):
    dst_df = src_df.copy()
    tmp_df = src_df['place_with_parent_names'].str.split("|", expand=True)

    dst_df['country'] = tmp_df[1]
    dst_df['region'] = tmp_df[2]
    dst_df['city'] = tmp_df[3]

    return dst_df

df_split = split_place_with_parent_names(df)
df_split

### Drop non-Brazil entries

In [ ]:
def drop_non_brazil_entries(src_df):
    return src_df[src_df["country"] == "Brasil"].copy()

df_brazil = drop_non_brazil_entries(df_split)
df_brazil

### Remove unused columns

In [ ]:
def remove_unused_colums(src_df):
    return src_df.drop(columns=[
        'place_name',
        'geonames_id',
        'place_with_parent_names',
        'lat-lon',
        'currency',
        'price_aprox_local_currency',
        'expenses',
        'properati_url',
        'description',
        'title',
        'image_thumbnail']
    )

df_less_columns = remove_unused_colums(df_brazil)
df_less_columns.info()

### Count # of NaN values

In [ ]:
df_less_columns.isna().sum()

### Drop entries without price

In [ ]:
def drop_entries_without_price(src_df):
    return src_df.dropna(subset=["price"]).copy()

df_priced = drop_entries_without_price(df_less_columns)
df_priced

In [ ]:
df_priced.info()

In [ ]:
df_priced.to_csv(f'data/silver/{DATASET_NAME}.csv', index=False)

## Analysis (Gold)

In [ ]:
df_gold = pd.read_csv(f"data/silver/{DATASET_NAME}.csv", parse_dates=["created_on"])
df_gold.info()

### Average Rent Price per Region

In [ ]:

avg_price_per_region = df_gold.groupby('region')['price_aprox_usd'].mean().sort_values(ascending=False)

plt.barh(avg_price_per_region.index, avg_price_per_region.values, color='skyblue', edgecolor='black')

plt.xlabel('Avg Price (U$)')
plt.ylabel('Regions')
plt.title('Average Renting Price per Region')

plt.savefig('data/gold/rent_avg_price_per_region.png')

### Average Price per m2 per Region

In [ ]:
avg_price_m2_by_region = df_gold.groupby('region')['price_usd_per_m2'].mean().sort_values(ascending=True)

plt.barh(avg_price_m2_by_region.index, avg_price_m2_by_region.values, color='lightgreen', edgecolor='black')

plt.title('Average Price per m2 by Region', fontsize=14)
plt.xlabel('Average Price per m2 (U$)', fontsize=12)
plt.ylabel('Region', fontsize=12)
plt.tight_layout()

plt.savefig('data/gold/rent_avg_price_m2_by_region.png')

### Average Price per Type and Type distribution

In [ ]:


type_counts = df_gold['property_type'].value_counts()

avg_price_by_type = df_gold.groupby('property_type')['price_aprox_usd'].mean().sort_values(ascending=False)

colors = ['#ff9999', '#66b3ff', "#66ffe0", "#f0ff66"]  # Soft pink/red for House, Soft blue for Apartment

plt.figure(figsize=(8,5))
plt.subplot(1, 2, 1)
plt.pie(
    type_counts, 
    labels=type_counts.index, 
    autopct='%1.1f%%',       # Shows percentages rounded to 1 decimal place
    startangle=90,           # Rotates the start of the pie chart for clean alignment
    colors=colors, 
    wedgeprops={'edgecolor': 'black'} # Adds clean borders to pie slices
)
plt.title('Type Distribution', fontsize=12)

x_pos = [2, 4, 6, 8]

plt.subplot(1, 2, 2)
plt.bar(
    x_pos, 
    avg_price_by_type.values, 
    color=colors, 
    edgecolor='black',
)
plt.xticks(x_pos, avg_price_by_type.index)
plt.title('Avg Price by Type', fontsize=12)
plt.xlabel('Property Type')
plt.ylabel('Avg Price (U$)')
plt.subplots_adjust(wspace=0.4)

plt.tight_layout()

plt.savefig('data/gold/rent_avg_price_by_type.png')

### Market price trends along the time (90 days window)

In [ ]:
trend_data = df_gold.groupby(['created_on', 'property_type'])['price_aprox_usd'].mean().unstack()

# Smooth out the daily transactional noise using a X-day rolling window
smooth_trends = trend_data.rolling(window=90, min_periods=1).mean()

# Plot the clear lines
plt.plot(smooth_trends.index, smooth_trends['house'], color='#ff9999', linewidth=2.5, label='Houses')
plt.plot(smooth_trends.index, smooth_trends['apartment'], color='#66b3ff', linewidth=2.5, label='Apartments')
plt.plot(smooth_trends.index, smooth_trends['store'], color="#66ffdb", linewidth=2.5, label='Store')
plt.plot(smooth_trends.index, smooth_trends['PH'], color="#f7ff66", linewidth=2.5, label='PH')

# Formatting enhancements
plt.title('Real Estate Market Price Trends (90 day window)', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Smoothed Average Price ($)')
plt.grid(True, linestyle='--', alpha=0.4)
plt.legend()
plt.xticks(rotation=30, ha='right')

# plt.tight_layout()
plt.savefig('data/gold/rent_market_price_trends.png')

### Price distribution per region (boxplot)

In [ ]:
clip_low, clip_high = df_gold['price_aprox_usd'].quantile([0.01, 0.99])
df_box = df_gold[df_gold['price_aprox_usd'].between(clip_low, clip_high)]

ordered_regions = df_box.groupby('region')['price_aprox_usd'].median().sort_values().index
data_by_region = [df_box[df_box['region'] == r]['price_aprox_usd'].values for r in ordered_regions]

plt.figure(figsize=(9, 6))
plt.boxplot(data_by_region, vert=False, tick_labels=ordered_regions, showfliers=False)

plt.title('Rent Price Distribution per Region', fontsize=14)
plt.xlabel('Price (U$)')
plt.ylabel('Region')
plt.grid(True, axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()

plt.savefig('data/gold/rent_price_distribution_boxplot.png')

### Listing volume vs price (monthly)

In [ ]:
monthly = df_gold.set_index('created_on').resample('ME').agg(
    count=('price_aprox_usd', 'size'),
    avg_price=('price_aprox_usd', 'mean'),
)
monthly['avg_price_smooth'] = monthly['avg_price'].rolling(3, min_periods=1).mean()

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(monthly.index, monthly['count'], width=20, color='#66b3ff', alpha=0.6, edgecolor='black', label='Listings / month')
ax1.set_xlabel('Date')
ax1.set_ylabel('Listings per month', color='#3877b5')
ax1.tick_params(axis='y', labelcolor='#3877b5')

ax2 = ax1.twinx()
ax2.plot(monthly.index, monthly['avg_price_smooth'], color='#ff6666', linewidth=2.5, label='Avg price (3-mo smoothed)')
ax2.set_ylabel('Avg price (U$)', color='#b65a5a')
ax2.tick_params(axis='y', labelcolor='#b65a5a')

plt.title('Rent: Listing Volume vs Avg Price (monthly)', fontsize=14)
fig.autofmt_xdate()
plt.tight_layout()

plt.savefig('data/gold/rent_volume_vs_price.png')

### Seasonality — avg price by month-of-year

In [ ]:
season = df_gold.copy()
season['month'] = season['created_on'].dt.month
season_pivot = season.groupby(['month', 'property_type'])['price_aprox_usd'].mean().unstack()

month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
type_colors = {'house': '#ff9999', 'apartment': '#66b3ff', 'store': '#66ffdb', 'PH': '#f7ff66'}

plt.figure(figsize=(9, 5))
for ptype in season_pivot.columns:
    plt.plot(
        season_pivot.index,
        season_pivot[ptype],
        color=type_colors.get(ptype, '#888888'),
        linewidth=2.5,
        marker='o',
        label=ptype,
    )

plt.title('Rent Seasonality — Avg Price by Month-of-Year', fontsize=14)
plt.xlabel('Month')
plt.ylabel('Avg Price (U$)')
plt.xticks(range(1, 13), month_labels)
plt.grid(True, linestyle='--', alpha=0.4)
plt.legend()
plt.tight_layout()

plt.savefig('data/gold/rent_seasonality.png')

### Apartment vs House — price per m² gap

In [ ]:
gap_data = df_gold.groupby(['region', 'property_type'])['price_usd_per_m2'].mean().unstack()
gap_data = gap_data.dropna(subset=['apartment', 'house'])
gap_data['gap'] = gap_data['apartment'] - gap_data['house']
gap_data = gap_data.sort_values('gap')

bar_colors = ['#ff9999' if v < 0 else '#66b3ff' for v in gap_data['gap']]

plt.figure(figsize=(9, 6))
plt.barh(gap_data.index, gap_data['gap'], color=bar_colors, edgecolor='black')
plt.axvline(0, color='black', linewidth=1)

plt.title('Rent: Apartment vs House — Price per m² Gap', fontsize=14)
plt.xlabel('Apartment − House (U$ per m²)')
plt.ylabel('Region')
plt.grid(True, axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()

plt.savefig('data/gold/rent_apartment_vs_house_gap.png')